# dataset_crop — trim a classifier dataset down to the detected object

Takes a classifier dataset laid out as one sub-folder per class, runs an object-detection
model on every image, and writes a new dataset with the same class sub-folders where each
image is cropped to the detected bounding box plus an offset.

Two `Detection` objects do the work, both on still image files (no server, no camera):

1. **`od`** — the object-detection model finds the object; its highest-confidence box is the crop region.
2. **`idle`** — a detection with no `cmd`: it only applies the ROI. The ROI is the box from step 1,
   `crop=1`, and `offset=OFFSET` px around it. `display.save_img_roi` writes the cropped image.

Images where nothing is detected are skipped and listed at the end.

## 1. Parameters

In [ ]:
# Classifier dataset: DATASET_DIR/<class_name>/*.jpg|png|...
DATASET_DIR = "/path/to/classifier_dataset"

# Object-detection model (.pkl exported by training_notebooks/train_object_detection.ipynb)
MODEL_PATH = "/path/to/object_detection.pkl"

# Where the cropped dataset goes; the class sub-folders are created inside it
OUTPUT_DIR = "/path/to/classifier_dataset_cropped"

# ROI offset in pixels, the same `offset` the playground's ROI slider uses:
# each box corner moves this far outward from the box centroid (negative shrinks),
# so a square box grows by ~0.7 * OFFSET on every side.
OFFSET = 100

# Detection confidence threshold, and an optional class filter:
#   CLS = None          -> every class the model knows
#   CLS = ["tube"]      -> only these labels count as "the object"
CONF = 0.5
CLS = None

IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

## 2. Build the two detections

In [ ]:
from dorna_vision import Detection

# highest-confidence box wins
od = Detection(
    detection={"cmd": "od", "path": MODEL_PATH, "conf": CONF, "cls": CLS},
    sort={"cmd": "conf", "ascending": False, "max_det": 1},
)

# no cmd: only the ROI stage runs, and save_img_roi writes what it produced
idle = Detection(
    detection={},
    roi={"corners": [], "inv": 0, "crop": 1, "offset": OFFSET},
    display={"label": 0, "save_img": 0, "save_img_roi": 0},
)

## 3. Walk the dataset

For each image: detect, take the box `corners`, then run `idle` on the same file with
`roi.corners = <box>` and `display.save_img_roi = <output path>`. Output keeps the class
sub-folder and the file name.

In [ ]:
import os

classes = sorted(d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d)))
print("classes:", classes)

done, skipped = {}, []
for cls_name in classes:
    src_dir = os.path.join(DATASET_DIR, cls_name)
    dst_dir = os.path.join(OUTPUT_DIR, cls_name)
    os.makedirs(dst_dir, exist_ok=True)
    done[cls_name] = 0

    for name in sorted(os.listdir(src_dir)):
        if not name.lower().endswith(IMG_EXT):
            continue
        src = os.path.join(src_dir, name)
        dst = os.path.join(dst_dir, name)

        # 1. object detection -> box
        found = od.run(data=src)
        if not found:
            skipped.append(src)
            continue
        bb = found[0]["corners"]

        # 2. idle detection: ROI = box + OFFSET, cropped, saved
        idle.run(
            data=src,
            roi={"corners": bb, "inv": 0, "crop": 1, "offset": OFFSET},
            display={"label": 0, "save_img": 0, "save_img_roi": dst},
        )
        done[cls_name] += 1

    print(f"{cls_name}: {done[cls_name]} cropped")

# save_img_roi writes on a background thread; close() joins them
idle.close()
od.close()

print("\ntotal cropped:", sum(done.values()))
print("skipped (no detection):", len(skipped))
for s in skipped:
    print("  ", s)

## 4. Preview one crop per class

In [ ]:
from IPython.display import Image, display

for cls_name in classes:
    dst_dir = os.path.join(OUTPUT_DIR, cls_name)
    files = sorted(f for f in os.listdir(dst_dir) if f.lower().endswith(IMG_EXT))
    if not files:
        continue
    print(cls_name, "->", files[0])
    display(Image(filename=os.path.join(dst_dir, files[0])))